# DDR intake: assemble, verify, inventory (no model touches DDR here)

**Purpose.** Get DDR into a known, hashed state and answer the questions lock v5 depends on, before any prediction exists:
1. image count, grade distribution (including grade 5 = ungradable), and the dataset's own train/valid/test assignment;
2. whether filenames encode a patient identifier, because patient-clustered intervals on DDR depend on it;
3. whether every labelled image exists and is readable, and whether any image appears twice (byte-identical).

**Setup before running.** Open the DDR Google Drive folder from the dataset's GitHub page, select all ten `DDR-dataset.zip.0*` chunks, and add them (Organize, then Add shortcut) to `dr_ensemble_v2/08_ddr`. CPU runtime is enough, but choose High-RAM if offered. Extraction happens on local Colab disk, not on Drive.

**Outputs.** `01_splits/ddr_manifest_v2.csv` (one row per labelled image: path, grade, source split, filename tokens, sha256) and `99_audit/ddr_intake_v2.json` (hashes of every chunk and the assembled zip, inventory, patient-ID analysis). No preprocessing, no inference.

In [1]:
# Cell 1: paths
from google.colab import drive
drive.mount('/content/drive')
import os, glob, hashlib, json, re, zipfile, datetime, subprocess
from collections import Counter
import pandas as pd

BASE   = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
DDR_IN = f'{BASE}/08_ddr'
SPLITS = f'{BASE}/01_splits'
AUDIT  = f'{BASE}/99_audit'
LOCAL  = '/content/ddr_local'
os.makedirs(LOCAL, exist_ok=True)

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()

chunks = sorted(glob.glob(f'{DDR_IN}/DDR-dataset.zip.0*'))
print(len(chunks), 'chunks found:'); [print(' ', os.path.basename(c), f'{os.path.getsize(c)/1e9:.2f} GB') for c in chunks]
assert len(chunks) == 10, 'Expected 10 chunks (DDR-dataset.zip.001 to .010). Check the shortcuts in 08_ddr.'

Mounted at /content/drive
10 chunks found:
  DDR-dataset.zip.001 1.57 GB
  DDR-dataset.zip.002 1.57 GB
  DDR-dataset.zip.003 1.57 GB
  DDR-dataset.zip.004 1.57 GB
  DDR-dataset.zip.005 1.57 GB
  DDR-dataset.zip.006 1.57 GB
  DDR-dataset.zip.007 1.57 GB
  DDR-dataset.zip.008 1.57 GB
  DDR-dataset.zip.009 1.57 GB
  DDR-dataset.zip.010 0.99 GB


In [2]:
# Cell 2: hash chunks, assemble, extract to local disk
CHUNK_SHA = {os.path.basename(c): sha256_file(c) for c in chunks}
ZIP = f'{LOCAL}/DDR-dataset.zip'
with open(ZIP, 'wb') as out:
    for c in chunks:
        with open(c, 'rb') as f:
            for blk in iter(lambda: f.read(1 << 24), b''): out.write(blk)
ZIP_SHA = sha256_file(ZIP)
print(f'Assembled {os.path.getsize(ZIP)/1e9:.2f} GB, sha256 {ZIP_SHA[:16]}...')
with zipfile.ZipFile(ZIP) as z:
    bad = z.testzip()
    assert bad is None, f'Corrupt member in archive: {bad}'
    z.extractall(LOCAL)
os.remove(ZIP)
print('Extracted. Top level:', sorted(os.listdir(LOCAL)))

Assembled 15.14 GB, sha256 a50cf62cf68031e7...
Extracted. Top level: ['DDR-dataset']


In [3]:
# Cell 3: find the grading label files and build the manifest
# DDR grading labels are text files of 'filename grade' lines. Located by content, not assumed path.
IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff')
line_re = re.compile(r'^\s*(\S+\.(?:jpg|jpeg|png|tif|tiff))\s+([0-5])\s*$', re.I)

label_files = []
for root, _, files in os.walk(LOCAL):
    for fn in files:
        if fn.lower().endswith('.txt'):
            p = os.path.join(root, fn)
            with open(p, errors='ignore') as f: head = [next(f, '') for _ in range(5)]
            if sum(bool(line_re.match(h)) for h in head) >= 3: label_files.append(p)
print('Grading label files:'); [print(' ', os.path.relpath(p, LOCAL)) for p in label_files]
assert label_files, 'No grading label files found; send me the printed directory tree.'

all_imgs = {}
for root, _, files in os.walk(LOCAL):
    for fn in files:
        if fn.lower().endswith(IMG_EXT): all_imgs.setdefault(fn, []).append(os.path.join(root, fn))

rows = []
for lf in label_files:
    src = os.path.splitext(os.path.basename(lf))[0].lower()   # train / valid / test
    with open(lf) as f:
        for line in f:
            m = line_re.match(line)
            if not m: continue
            name, g = m.group(1), int(m.group(2))
            # Prefer the copy that sits beside the label file (grading folder), else any match.
            cands = all_imgs.get(name, [])
            here = ([p for p in cands if f'{os.sep}{src}{os.sep}' in p] or
                    [p for p in cands if os.path.dirname(lf) in p])
            path = (here or all_imgs.get(name, [None]))[0]
            rows.append({'image_id': os.path.splitext(name)[0], 'filename': name, 'grade': g,
                         'source_split': src, 'path_local': path})
man = pd.DataFrame(rows)
print(f'{len(man):,} labelled rows from {len(label_files)} files; missing images: {man.path_local.isna().sum()}')
print('Duplicate filenames across label files:', int(man.filename.duplicated().sum()))
print(man.groupby(['source_split','grade']).size().unstack(fill_value=0))

Grading label files:
  DDR-dataset/DR_grading/test.txt
  DDR-dataset/DR_grading/valid.txt
  DDR-dataset/DR_grading/train.txt
13,673 labelled rows from 3 files; missing images: 0
Duplicate filenames across label files: 0
grade            0    1     2    3    4    5
source_split                                
test          1880  189  1344   71  275  346
train         3133  315  2238  118  456  575
valid         1253  126   895   47  182  230


In [4]:
# Cell 4: patient-identifier analysis from filename structure
# The DDR paper reports 13,673 images from 9,598 patients. If a filename token (or prefix) yields ~9,598 groups
# with at most a handful of images each, it is a candidate patient key. Reported, not assumed.
stems = man.image_id.astype(str)
toks = stems.str.split(r'[-_]', regex=True)
max_len = int(toks.str.len().max())
print('Filename token counts:', Counter(toks.str.len()).most_common())
print('Example stems:', stems.sample(8, random_state=0).tolist())

candidates = []
for k in range(1, max_len):
    key = toks.apply(lambda t: '-'.join(t[:k]))
    n = key.nunique(); per = key.value_counts()
    candidates.append({'prefix_tokens': k, 'n_groups': int(n), 'max_images_per_group': int(per.max()),
                       'pct_groups_le_4_images': round(float((per <= 4).mean()), 4)})
for i in range(max_len):
    key = toks.str[i]
    candidates.append({'single_token_index': i, 'n_groups': int(key.nunique())})
cand_df = pd.DataFrame(candidates)
print(cand_df.to_string(index=False))
print('\nReference: 9,598 patients reported in the DDR paper. A good key gives close to that, with <= ~4 images per group.')

Filename token counts: [(3, 8250), (1, 5423)]
Example stems: ['20170420095437998', '20170327105554324', '20170215164948886', '007-3569-200', '007-7600-601', '007-5860-300', '20170213081351910', '007-2908-100']
 prefix_tokens  n_groups  max_images_per_group  pct_groups_le_4_images  single_token_index
           1.0      5424                8250.0                  0.9998                 NaN
           2.0     13673                   1.0                  1.0000                 NaN
           NaN      5424                   NaN                     NaN                 0.0
           NaN      8250                   NaN                     NaN                 1.0
           NaN        10                   NaN                     NaN                 2.0

Reference: 9,598 patients reported in the DDR paper. A good key gives close to that, with <= ~4 images per group.


In [5]:
# Cell 5: readability and byte-level duplicates (all labelled images)
import numpy as np
from PIL import Image
sha, unread = [], []
for p in man.path_local:
    if p is None or not os.path.exists(p): sha.append(None); continue
    sha.append(sha256_file(p))
    try:
        with Image.open(p) as im: im.verify()
    except Exception: unread.append(p)
man['sha256'] = sha
dup_groups = man.dropna(subset=['sha256']).groupby('sha256').filter(lambda d: len(d) > 1)
print(f'Unreadable: {len(unread)}; byte-identical images appearing more than once: {dup_groups.sha256.nunique()} groups, {len(dup_groups)} rows')
if len(dup_groups): print(dup_groups.sort_values('sha256')[['filename','grade','source_split']].head(20).to_string(index=False))

Unreadable: 0; byte-identical images appearing more than once: 97 groups, 197 rows
             filename  grade source_split
20170430141737870.jpg      0         test
20170428105426121.jpg      0         test
20170221162224605.jpg      0        train
20170329094415243.jpg      0        valid
20170329143238497.jpg      0        valid
20170324140245887.jpg      0        valid
20170420095437998.jpg      0         test
20170421092005051.jpg      0         test
20170424153038492.jpg      0         test
20170424093951909.jpg      0         test
20170226093027863.jpg      0        train
20170226083551331.jpg      0        train
20170222150740420.jpg      0        train
20170223171302321.jpg      0        train
20170204114217232.jpg      0        train
20170204114728704.jpg      0        train
20170329143208143.jpg      0        valid
20170324140208565.jpg      0        valid
20170323083645071.jpg      0        train
20170323085108829.jpg      0        train


In [6]:
# Cell 6: write manifest and intake record
man_out = man.drop(columns=['path_local']).copy()
man_out['relpath'] = [os.path.relpath(p, LOCAL) if isinstance(p, str) else None for p in man.path_local]
man_out.to_csv(f'{SPLITS}/ddr_manifest_v2.csv', index=False)

gc = man.grade.value_counts().sort_index()
gradable = man[man.grade <= 4]
record = {
  'written_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'source': 'nkicsl/DDR-dataset (OIA-DDR), Google Drive mirror; Li et al., Information Sciences 2019;501:511-522',
  'chunk_sha256': CHUNK_SHA, 'assembled_zip_sha256': ZIP_SHA,
  'label_files': [os.path.relpath(p, LOCAL) for p in label_files],
  'n_labelled': int(len(man)), 'n_missing_image': int(man.path_local.isna().sum()),
  'n_unreadable': len(unread), 'byte_duplicate_groups': int(dup_groups.sha256.nunique()),
  'grade_counts_all': {str(k): int(v) for k, v in gc.items()},
  'n_gradable': int(len(gradable)),
  'referable_prevalence_gradable': round(float((gradable.grade >= 2).mean()), 4),
  'pdr_eyes_gradable': int((gradable.grade == 4).sum()),
  'by_source_split': man.groupby(['source_split','grade']).size().unstack(fill_value=0).to_dict(),
  'patient_key_candidates': cand_df.to_dict(orient='records'),
  'manifest_sha256': sha256_file(f'{SPLITS}/ddr_manifest_v2.csv'),
  'note': 'Intake only. No preprocessing, no inference. Lock v5 is written after this record and before any DDR prediction exists.'}
with open(f'{AUDIT}/ddr_intake_v2.json', 'w') as f: json.dump(record, f, indent=2)
with open(f'{AUDIT}/RUN_LOG.md', 'a') as f:
    f.write(f'''\n\n## DDR intake ({record['written_utc'][:10]})\n\nAssembled zip sha256 {ZIP_SHA[:16]}...; {len(man):,} labelled images, '''
            f'''{len(gradable):,} gradable, {record['pdr_eyes_gradable']} PDR, referable prevalence {record['referable_prevalence_gradable']}. '''
            f'''Manifest 01_splits/ddr_manifest_v2.csv. No model has seen DDR. Produced by 99_audit/DDR_intake_v2.ipynb.\n''')
print(json.dumps({k: v for k, v in record.items() if k not in ('by_source_split','patient_key_candidates','chunk_sha256')}, indent=2))

{
  "written_utc": "2026-09-23T13:57:05.015242+00:00",
  "source": "nkicsl/DDR-dataset (OIA-DDR), Google Drive mirror; Li et al., Information Sciences 2019;501:511-522",
  "assembled_zip_sha256": "a50cf62cf68031e7ebde6312e17a3c65ce5f7aa2294f7f4c3484bf1ac54e9178",
  "label_files": [
    "DDR-dataset/DR_grading/test.txt",
    "DDR-dataset/DR_grading/valid.txt",
    "DDR-dataset/DR_grading/train.txt"
  ],
  "n_labelled": 13673,
  "n_missing_image": 0,
  "n_unreadable": 0,
  "byte_duplicate_groups": 97,
  "grade_counts_all": {
    "0": 6266,
    "1": 630,
    "2": 4477,
    "3": 236,
    "4": 913,
    "5": 1151
  },
  "n_gradable": 12522,
  "referable_prevalence_gradable": 0.4493,
  "pdr_eyes_gradable": 913,
  "manifest_sha256": "9c6b9cd1bf1872d9c6c84b173b1fabc5f03c8942754f2ee66fc3fadffe95848a",
  "note": "Intake only. No preprocessing, no inference. Lock v5 is written after this record and before any DDR prediction exists."
}


## After this notebook
Send back `99_audit/ddr_intake_v2.json` and move this executed notebook to `99_audit/DDR_intake_v2.ipynb`. The patient-key table decides whether DDR intervals can be patient-clustered. Lock v5 (the external replication) is written from this record, before preprocessing and inference.